In [1]:
import polars as pl
from datasets import load_dataset, load_from_disk
import pandas as pd
from replay.metrics import Recall, Precision, HitRate

In [2]:
ranking_candidates = pl.read_parquet("../all_candidatates_recs.parquet")

In [3]:
DATA_PATH = "/home/jupyter/filestore/storage/datasets/user_clicks_20230501"

dataset = load_from_disk(DATA_PATH)

In [4]:
polars_ds = dataset.to_polars()

In [5]:
test = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") > TRAIN_END_DT)
    .filter(pl.col("date") <=  TEST_END_DT)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("future_clicks"))
)

NameError: name 'TRAIN_END_DT' is not defined

In [6]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

test_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") > TRAIN_END_DT)
    .filter(pl.col("date") <=  TEST_END_DT)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("future_clicks"))
    .join(
        ranking_candidates,
        on="user_id",
        how="inner"
    )
)

In [8]:
test_interactions.shape

(115058, 3)

In [8]:
def calc_hitrate(row):
    return HitRate._get_metric_value_by_user([1000], row["future_clicks"], row["recs"])

def get_positive_candidates(row):
    return list(set(row["future_clicks"]) & set(row["recs"]))

def positive_recs_cnt(row):
    return len(list(set(row["future_clicks"]) & set(row["recs"])))
                
def get_negative_candidates(row, k=5):
    positive_candidates_cnt = len(row["positive_candidates"])
    negatives_candidates_cnt = min(6 * positive_candidates_cnt, 100)
    return list(set(row["recs"]) - set(row["positive_candidates"]))[:negatives_candidates_cnt]


candidates_for_ranking = (
    test_interactions
    .with_columns(
        pl.struct(["future_clicks", "recs"]).apply(calc_hitrate).arr.get(0).alias("hitrate"),
    )
    .filter(pl.col("hitrate") == 1)
    .with_columns(
        pl.struct(["future_clicks", "recs"]).apply(get_positive_candidates).alias("positive_candidates")
    )
    .with_columns(
        pl.struct(["positive_candidates", "recs"]).apply(get_negative_candidates).alias("negative_candidates")
    )
)

In [37]:
candidates_for_ranking.head(5)

user_id,future_clicks,recs,hitrate,positive_candidates,negative_candidates
i64,list[i64],list[i64],f64,list[i64],list[i64]
794,"[211742298, 182628793, … 184880949]","[35946499, 226879493, … 137959423]",1.0,[211742298],"[35946499, 226879493, … 162863117]"
985,"[157059482, 119989884, … 33560808]","[189161472, 236388360, … 18479097]",1.0,[50952690],"[189161472, 236388360, … 73357326]"
3136,"[164142726, 136811254, 213488536]","[219521026, 140859404, … 16670717]",1.0,"[164142726, 136811254]","[219521026, 140859404, … 164485153]"
5363,"[66290793, 233996790, … 112608394]","[217018368, 130265088, … 181770235]",1.0,[233996790],"[217018368, 130265088, … 135768065]"
6279,"[135008976, 159543907, … 184386075]","[89374720, 43393024, … 134619132]",1.0,[209787866],"[89374720, 43393024, … 194103302]"


In [9]:
positive_df = (
    candidates_for_ranking
    .explode("positive_candidates")
    .select(
        pl.col("user_id"),
        pl.col("positive_candidates").alias("item_id"),
        pl.lit(1).alias("target")
    )
)

negative_df = (
    candidates_for_ranking
    .explode("negative_candidates")
    .select(
        pl.col("user_id"),
        pl.col("negative_candidates").alias("item_id"),
        pl.lit(0).alias("target")
    )
)

In [10]:
train_dataset = pl.concat([positive_df, negative_df])

In [16]:
train_dataset.shape

(1425658, 3)

In [17]:
train_dataset.write_parquet("/home/jupyter/filestore/storage/datasets/train_ranking_dataset.parquet")